# ICCiT 2026 — Final Paper-Ready Benchmark

## Edge-Oriented Binary Conjunctivitis Classification from Anterior-Segment Images

This notebook is designed as the **candidate final experimental source of truth** for the manuscript.

### Fixed research story
- **Primary edge-oriented reference:** MobileNetV3-Small
- **Secondary higher-capacity reference:** ConvNeXt-Tiny
- **High-capacity transformer reference:** ViT-B/16
- **Cross-paradigm context:** MobileNetV2, EfficientNet-B0, DenseNet201, ResNet50, VGG16, Swin-T

### Scientific rules
1. Preserve the original backbone identities: torchvision models remain torchvision; timm models remain timm.
2. Use a common binary classifier-head topology across all nine models, without silently replacing any backbone implementation.
3. Use one fixed train/validation/test split and never use the held-out test set for tuning.
4. Determine the operating threshold from validation data only using Youden's J statistic.
5. Select checkpoints using validation loss.
6. Report bootstrap confidence intervals and exact paired McNemar tests with Holm correction.
7. Never substitute fabricated runtime values when ONNX export or inference fails.
8. MobileNetV3-Small optimization is a separate validation-driven experiment; it does not overwrite the standardized benchmark.
9. Physical smartphone/ARM deployment is **not** claimed; ONNX CPU is an edge-oriented computational proxy.

> **Important:** Run the sanity-check cell before the full benchmark. Do not change the split, benchmark protocol, or model roles after inspecting final test results.

In [1]:
# ==============================================================================
# CELL 1 — ENVIRONMENT, CONFIGURATION, VERSION LOG, AND REPRODUCIBILITY
# ===============================================================================

from __future__ import annotations

import hashlib
import json
import os
import platform
import random
import subprocess
import sys
import time
from copy import deepcopy
from pathlib import Path

# Environment controls must be set before CUDA work.
os.environ.setdefault("PYTHONHASHSEED", "42")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

# Optional package bootstrap for Kaggle. These packages are required for the final paper outputs.
REQUIRED_EXTRA = ["onnx", "onnxruntime", "timm", "albumentations"]
missing = []
for pkg in REQUIRED_EXTRA:
    try:
        __import__(pkg)
    except ImportError:
        missing.append(pkg)

if missing:
    print("Missing packages:", missing)
    print("Attempting one-time Kaggle installation...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import cv2
import numpy as np
import pandas as pd
from PIL import Image

import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import models

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split

import onnx
import onnxruntime as ort

# ---------------------- Pre-specified experiment configuration ----------------------
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2
DROP_LAST = False

STAGE1_EPOCHS = 20
STAGE1_PATIENCE = 8
STAGE1_LR = 1e-3
STAGE2_EPOCHS = 15
STAGE2_PATIENCE = 8
STAGE2_LR = 1e-5
WEIGHT_DECAY = 1e-4
FOCAL_GAMMA = 2.0

PRIMARY_MODEL = "mobilenet_v3_small"
SECONDARY_MODEL = "convnext_tiny"
HIGH_CAPACITY_REFERENCE = "vit_base_patch16_224"
REFERENCE_MODEL = PRIMARY_MODEL

BENCHMARK_MODELS = [
    "mobilenet_v3_small",
    "mobilenet_v2",
    "efficientnet_b0",
    "densenet201",
    "resnet50",
    "vgg16",
    "convnext_tiny",
    "swin_tiny_patch4_window7_224",
    "vit_base_patch16_224",
]

MODEL_DISPLAY_NAMES = {
    "mobilenet_v3_small": "MobileNetV3-Small",
    "mobilenet_v2": "MobileNetV2",
    "efficientnet_b0": "EfficientNet-B0",
    "densenet201": "DenseNet201",
    "resnet50": "ResNet50",
    "vgg16": "VGG16",
    "convnext_tiny": "ConvNeXt-Tiny",
    "swin_tiny_patch4_window7_224": "Swin-T",
    "vit_base_patch16_224": "ViT-B/16",
}

# Runtime profiling is part of the final paper table.
PROFILE_GPU = True
PROFILE_PYTORCH_CPU = True
PROFILE_ONNX = True
GPU_WARMUPS = 30
GPU_ITERS = 100
CPU_WARMUPS = 30
CPU_ITERS = 100
ONNX_WARMUPS = 30
ONNX_ITERS = 100
CPU_THREADS = 2
ONNX_OPSET = 17
STRICT_ONNX = True

# Optional secondary experiments. Main benchmark is still valid with these False.
RUN_MOBILENET_OPTIMIZATION = False
MOBILENET_OPTIMIZATION_SEEDS = [42, 7, 123]
RUN_XAI = False

# Publication title / wording used in exported metadata.
PAPER_TITLE = (
    "Edge-Oriented Binary Conjunctivitis Classification from Anterior-Segment "
    "Images: A Cross-Paradigm Deep Learning Benchmark"
)

WORK_DIR = Path("/kaggle/working")
AUDIT_DIR = WORK_DIR / "audit_reports"
SPLIT_DIR = WORK_DIR / "splits"
MODEL_DIR = WORK_DIR / "models"
RESULT_DIR = WORK_DIR / "results"
PLOT_DIR = WORK_DIR / "plots"
TABLE_DIR = WORK_DIR / "tables"
META_DIR = WORK_DIR / "metadata"
CLEAN_DIR = WORK_DIR / "dataset_cleaned"

for d in [AUDIT_DIR, SPLIT_DIR, MODEL_DIR, RESULT_DIR, PLOT_DIR, TABLE_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"


def seed_everything(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

version_info = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "timm": timm.__version__,
    "albumentations": A.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "opencv": cv2.__version__,
    "onnx": onnx.__version__,
    "onnxruntime": ort.__version__,
    "device": str(DEVICE),
}

run_config = {
    "paper_title": PAPER_TITLE,
    "seed": SEED,
    "benchmark_models": BENCHMARK_MODELS,
    "reference_model": REFERENCE_MODEL,
    "batch_size": BATCH_SIZE,
    "drop_last": DROP_LAST,
    "stage1": {"epochs": STAGE1_EPOCHS, "patience": STAGE1_PATIENCE, "lr": STAGE1_LR},
    "stage2": {"epochs": STAGE2_EPOCHS, "patience": STAGE2_PATIENCE, "lr": STAGE2_LR},
    "weight_decay": WEIGHT_DECAY,
    "focal_gamma": FOCAL_GAMMA,
    "profile_gpu": PROFILE_GPU,
    "profile_pytorch_cpu": PROFILE_PYTORCH_CPU,
    "profile_onnx": PROFILE_ONNX,
    "strict_onnx": STRICT_ONNX,
    "mobilenet_optimization_enabled": RUN_MOBILENET_OPTIMIZATION,
    "xai_enabled": RUN_XAI,
    "versions": version_info,
}

(META_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2))
print(json.dumps(run_config, indent=2))

Missing packages: ['onnxruntime']
Attempting one-time Kaggle installation...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 68.0 MB/s eta 0:00:00
{
  "paper_title": "Edge-Oriented Binary Conjunctivitis Classification from Anterior-Segment Images: A Cross-Paradigm Deep Learning Benchmark",
  "seed": 42,
  "benchmark_models": [
    "mobilenet_v3_small",
    "mobilenet_v2",
    "efficientnet_b0",
    "densenet201",
    "resnet50",
    "vgg16",
    "convnext_tiny",
    "swin_tiny_patch4_window7_224",
    "vit_base_patch16_224"
  ],
  "reference_model": "mobilenet_v3_small",
  "batch_size": 32,
  "drop_last": false,
  "stage1": {
    "epochs": 20,
    "patience": 8,
    "lr": 0.001
  },
  "stage2": {
    "epochs": 15,
    "patience": 8,
    "lr": 1e-05
  },
  "weight_decay": 0.0001,
  "focal_gamma": 2.0,
  "profile_gpu": true,
  "profile_pytorch_cpu": true,
  "profile_onnx": true,
  "strict_onnx": true,
  "mobilenet_optimization_enabled": false,
  "xai_enabled": false,
  "versions

In [2]:
# ==============================================================================
# CELL 2 — COHORT CURATION, DUPLICATE AUDIT, AND DATASET FREEZE
# ==============================================================================

LABEL_MAP = {"Non_Conjunctivitis": 0, "Conjunctivitis": 1}
MIN_RESOLUTION = 100
PHASH_HAMMING_THRESHOLD = 4

DISEASE_FOLDERS = {"conjunctivitis", "infected_eye"}
HEALTHY_FOLDERS = {
    "non_conjunctivitis", "non-conjunctivitis", "healthy_eye",
    "normal", "healthy",
}
EXCLUDE_KEYWORDS = {
    "augmented", "cataract", "glaucoma", "uveitis", "diabetic",
    "retinopathy", "pterygium", "oct", "fundus", "retina", "bscan", "keratitis",
}
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def md5_file(path: Path) -> str:
    h = hashlib.md5()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def phash(path: Path, hash_size=8, highfreq_factor=4):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    target = hash_size * highfreq_factor
    img = cv2.resize(img, (target, target), interpolation=cv2.INTER_AREA)
    dct = cv2.dct(np.float32(img))[:hash_size, :hash_size]
    return (dct > np.median(dct)).flatten()


def excluded_path(path: Path) -> bool:
    parts = [p.lower() for p in path.parts]
    joined = " ".join(parts)
    if any(k in joined for k in EXCLUDE_KEYWORDS):
        return True
    return path.name.lower().startswith(("aug_", "aug-", "augmented_"))

records = []
for root in sorted(p for p in Path("/kaggle/input").iterdir() if p.is_dir()):
    for p in sorted(root.rglob("*")):
        if p.suffix.lower() not in IMAGE_EXTS or excluded_path(p):
            continue
        parent = p.parent.name.lower()
        if parent in DISEASE_FOLDERS:
            cls = "Conjunctivitis"
        elif parent in HEALTHY_FOLDERS:
            cls = "Non_Conjunctivitis"
        else:
            continue
        try:
            with Image.open(p) as img:
                width, height = img.size
                mode, fmt = img.mode, img.format
        except Exception:
            width = height = 0
            mode, fmt = "CORRUPT", "UNKNOWN"
        records.append({
            "path": str(p),
            "source": root.name,
            "class": cls,
            "label": LABEL_MAP[cls],
            "width": width,
            "height": height,
            "min_dim": min(width, height),
            "mode": mode,
            "format": fmt,
            "md5": md5_file(p),
            "phash": phash(p),
        })

df_raw = pd.DataFrame(records)
if df_raw.empty:
    raise RuntimeError("No candidate images found under /kaggle/input.")

# Deterministic exact-byte duplicate handling with explicit label-conflict audit.
md5_remove = set()
md5_conflicts = []
for md5_value, g in df_raw.sort_values("path").groupby("md5"):
    if len(g) <= 1:
        continue
    labels = set(g["class"])
    if len(labels) > 1:
        md5_remove.update(g["path"])
        md5_conflicts.append({
            "md5": md5_value,
            "records": g[["path", "class"]].to_dict("records"),
        })
    else:
        md5_remove.update(g["path"].iloc[1:])

df_after_md5 = df_raw[~df_raw["path"].isin(md5_remove)].reset_index(drop=True)

# Pairwise pHash audit reproduces the original threshold but records conflicts explicitly.
phash_remove = set()
phash_conflicts = []
for i in range(len(df_after_md5)):
    pi = df_after_md5.loc[i, "path"]
    if pi in phash_remove:
        continue
    hi = df_after_md5.loc[i, "phash"]
    if hi is None:
        continue
    for j in range(i + 1, len(df_after_md5)):
        hj = df_after_md5.loc[j, "phash"]
        if hj is None:
            continue
        pj = df_after_md5.loc[j, "path"]
        if pj in phash_remove:
            continue
        if np.count_nonzero(hi != hj) <= PHASH_HAMMING_THRESHOLD:
            ci = df_after_md5.loc[i, "class"]
            cj = df_after_md5.loc[j, "class"]
            if ci != cj:
                phash_remove.update({pi, pj})
                phash_conflicts.append({
                    "path_a": pi, "class_a": ci,
                    "path_b": pj, "class_b": cj,
                })
            else:
                phash_remove.add(pj)

resolution_remove = set(
    df_after_md5[
        (df_after_md5["min_dim"] < MIN_RESOLUTION) |
        (df_after_md5["mode"] == "CORRUPT")
    ]["path"]
)

all_remove = md5_remove | phash_remove | resolution_remove
df_clean = df_raw[~df_raw["path"].isin(all_remove)].copy().reset_index(drop=True)

# Freeze clean image copies and write immutable-style manifests.
if CLEAN_DIR.exists():
    import shutil
    shutil.rmtree(CLEAN_DIR)
for cls in LABEL_MAP:
    (CLEAN_DIR / cls).mkdir(parents=True, exist_ok=True)

clean_rows = []
used_destinations = set()
for _, row in df_clean.iterrows():
    src = Path(row["path"])
    base_name = f"{row['source'][:6]}_{src.name}"
    dst = CLEAN_DIR / row["class"] / base_name
    if str(dst) in used_destinations or dst.exists():
        dst = CLEAN_DIR / row["class"] / f"{row['source'][:6]}_{src.stem}_{row['md5'][:8]}.jpg"
    used_destinations.add(str(dst))
    with Image.open(src) as img:
        img.convert("RGB").save(dst, "JPEG", quality=95)
    clean_rows.append({
        "cleaned_path": str(dst),
        "original_path": str(src),
        "source": row["source"],
        "class": row["class"],
        "label": int(row["label"]),
        "md5": row["md5"],
        "min_dim": int(row["min_dim"]),
    })

df_clean_manifest = pd.DataFrame(clean_rows)
df_clean_manifest.to_csv(AUDIT_DIR / "cleaned_dataset_manifest.csv", index=False)

manifest_rows = []
for _, row in df_raw.iterrows():
    reasons = []
    if row["path"] in md5_remove:
        reasons.append("Exact_MD5_or_MD5_Label_Conflict")
    if row["path"] in phash_remove:
        reasons.append("pHash_or_Conflict")
    if row["path"] in resolution_remove:
        reasons.append("Sub_100px_or_Corrupt")
    manifest_rows.append({
        "path": row["path"],
        "class": row["class"],
        "status": "EXCLUDED" if reasons else "RETAINED",
        "exclusion_reasons": ";".join(reasons) if reasons else "None",
        "md5": row["md5"],
        "min_dim": int(row["min_dim"]),
    })
pd.DataFrame(manifest_rows).to_csv(AUDIT_DIR / "full_curation_audit_manifest.csv", index=False)
pd.DataFrame(phash_conflicts).to_csv(AUDIT_DIR / "phash_conflicts.csv", index=False)
(AUDIT_DIR / "md5_label_conflicts.json").write_text(json.dumps(md5_conflicts, indent=2))

summary = {
    "raw_candidate_count": int(len(df_raw)),
    "exact_md5_exclusions": int(len(md5_remove)),
    "phash_or_conflict_exclusions": int(len(phash_remove)),
    "resolution_exclusions": int(len(resolution_remove)),
    "unique_exclusions_union": int(len(all_remove)),
    "clean_cohort_count": int(len(df_clean_manifest)),
    "class_counts": df_clean_manifest["class"].value_counts().to_dict(),
    "md5_label_conflict_groups": int(len(md5_conflicts)),
    "phash_conflict_pairs": int(len(phash_conflicts)),
}
(AUDIT_DIR / "curation_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

# Hard guard: unexpected cohort drift must be audited before model training.
EXPECTED_COHORT = 544
EXPECTED_COUNTS = {"Conjunctivitis": 388, "Non_Conjunctivitis": 156}
if len(df_clean_manifest) != EXPECTED_COHORT or df_clean_manifest["class"].value_counts().to_dict() != EXPECTED_COUNTS:
    raise RuntimeError(
        "Cohort differs from the manuscript's frozen cohort. "
        "Stop and audit curation before proceeding. "
        f"Observed={len(df_clean_manifest)}, counts={df_clean_manifest['class'].value_counts().to_dict()}"
    )

{
  "raw_candidate_count": 1344,
  "exact_md5_exclusions": 373,
  "phash_or_conflict_exclusions": 32,
  "resolution_exclusions": 397,
  "unique_exclusions_union": 800,
  "clean_cohort_count": 544,
  "class_counts": {
    "Conjunctivitis": 388,
    "Non_Conjunctivitis": 156
  },
  "md5_label_conflict_groups": 0,
  "phash_conflict_pairs": 1
}


In [3]:
# ==============================================================================
# CELL 3 — FROZEN STRATIFIED SPLIT + INTEGRITY CHECKS
# ==============================================================================

from sklearn.model_selection import train_test_split

df = pd.read_csv(AUDIT_DIR / "cleaned_dataset_manifest.csv").reset_index(drop=True)

# The current cohort has no verified patient/session identifiers in the public sources.
# Therefore the final experiment uses the documented image-level stratified split.
# If verified group IDs become available, replace this with a group-aware split before freezing.
GROUP_COLUMN = "group_id"
if GROUP_COLUMN in df.columns and df[GROUP_COLUMN].notna().all():
    raise NotImplementedError(
        "Verified group IDs detected. Implement and freeze a group-aware split before the final run."
    )

train_val, test = train_test_split(
    df,
    test_size=0.20,
    stratify=df["label"],
    random_state=SEED,
)
train, val = train_test_split(
    train_val,
    test_size=(0.10 / 0.80),
    stratify=train_val["label"],
    random_state=SEED,
)
train = train.sort_values("cleaned_path").reset_index(drop=True)
val = val.sort_values("cleaned_path").reset_index(drop=True)
test = test.sort_values("cleaned_path").reset_index(drop=True)

train.to_csv(SPLIT_DIR / "train_split.csv", index=False)
val.to_csv(SPLIT_DIR / "val_split.csv", index=False)
test.to_csv(SPLIT_DIR / "test_split.csv", index=False)


def assert_pairwise_disjoint(a, b, name_a, name_b):
    overlap_paths = set(a.cleaned_path) & set(b.cleaned_path)
    assert not overlap_paths, f"{name_a}/{name_b} overlap: {len(overlap_paths)}"
    overlap_md5 = set(a.md5) & set(b.md5)
    assert not overlap_md5, f"{name_a}/{name_b} MD5 overlap: {len(overlap_md5)}"

assert_pairwise_disjoint(train, val, "train", "val")
assert_pairwise_disjoint(train, test, "train", "test")
assert_pairwise_disjoint(val, test, "val", "test")

split_summary = {
    "train_n": int(len(train)),
    "val_n": int(len(val)),
    "test_n": int(len(test)),
    "train_class_counts": train["class"].value_counts().to_dict(),
    "val_class_counts": val["class"].value_counts().to_dict(),
    "test_class_counts": test["class"].value_counts().to_dict(),
    "split_level": "image-level; patient-level independence not verifiable for public repositories",
}
(SPLIT_DIR / "split_summary.json").write_text(json.dumps(split_summary, indent=2))
print(json.dumps(split_summary, indent=2))

{
  "train_n": 380,
  "val_n": 55,
  "test_n": 109,
  "train_class_counts": {
    "Conjunctivitis": 271,
    "Non_Conjunctivitis": 109
  },
  "val_class_counts": {
    "Conjunctivitis": 39,
    "Non_Conjunctivitis": 16
  },
  "test_class_counts": {
    "Conjunctivitis": 78,
    "Non_Conjunctivitis": 31
  },
  "split_level": "image-level; patient-level independence not verifiable for public repositories"
}


In [4]:
# ==============================================================================
# CELL 4 — API-SAFE AUGMENTATION + DATALOADERS
# ==============================================================================

import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader, Dataset

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def make_coarse_dropout(p=0.30):
    # Compatible with both newer and older Albumentations APIs.
    try:
        return A.CoarseDropout(
            num_holes_range=(1, 4),
            hole_height_range=(8, 16),
            hole_width_range=(8, 16),
            fill=0,
            p=p,
        )
    except TypeError:
        return A.CoarseDropout(
            max_holes=4,
            max_height=16,
            max_width=16,
            fill_value=0,
            p=p,
        )


def make_train_transform(full_aug=True):
    ops = [
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.HorizontalFlip(p=0.5),
        A.Affine(
            scale=(0.90, 1.10),
            rotate=(-15, 15),
            translate_percent=(-0.08, 0.08),
            interpolation=cv2.INTER_LINEAR,
            border_mode=cv2.BORDER_REFLECT,
            p=0.5,
        ),
        A.RandomBrightnessContrast(
            brightness_limit=0.15,
            contrast_limit=0.15,
            p=0.5,
        ),
    ]
    if full_aug:
        ops.extend([
            A.HueSaturationValue(
                hue_shift_limit=8,
                sat_shift_limit=15,
                val_shift_limit=15,
                p=0.25,
            ),
            A.CLAHE(clip_limit=2.0, p=0.20),
            make_coarse_dropout(p=0.20),
        ])
    ops.extend([
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])
    return A.Compose(ops)


eval_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

class OcularDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        path = self.frame.loc[idx, "cleaned_path"]
        label = float(self.frame.loc[idx, "label"])
        bgr = cv2.imread(path)
        if bgr is None:
            raise RuntimeError(f"Image read failure: {path}")
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        tensor = self.transform(image=rgb)["image"]
        return tensor, torch.tensor(label, dtype=torch.float32)


def make_loaders(train_df, val_df, test_df, full_aug=True, seed=SEED):
    seed_everything(seed)
    generator = torch.Generator()
    generator.manual_seed(seed)

    def worker_init_fn(worker_id):
        worker_seed = seed + worker_id
        random.seed(worker_seed)
        np.random.seed(worker_seed)
        torch.manual_seed(worker_seed)

    train_loader = DataLoader(
        OcularDataset(train_df, make_train_transform(full_aug=full_aug)),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        worker_init_fn=worker_init_fn,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=(NUM_WORKERS > 0),
        drop_last=DROP_LAST,
    )
    val_loader = DataLoader(
        OcularDataset(val_df, eval_transform),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=(NUM_WORKERS > 0),
    )
    test_loader = DataLoader(
        OcularDataset(test_df, eval_transform),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=(NUM_WORKERS > 0),
    )
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = make_loaders(train, val, test, full_aug=True, seed=SEED)

print("Train images/epoch:", len(train_loader.dataset), "drop_last=", DROP_LAST)
print("Train batches:", len(train_loader))
print("Validation images:", len(val_loader.dataset))
print("Test images:", len(test_loader.dataset))

Train images/epoch: 380 drop_last= False
Train batches: 12
Validation images: 55
Test images: 109


In [5]:
# ==============================================================================
# CELL 5 — NINE BACKBONES, ORIGINAL LIBRARIES, COMMON CLASSIFIER-HEAD TOPOLOGY
# ==============================================================================

COMMON_HEAD_DESCRIPTION = (
    "BatchNorm1d(in_features) -> Linear(in_features,512) -> ReLU -> "
    "Dropout(0.5) -> Linear(512,256) -> Dropout(0.3) -> Linear(256,1)"
)

class CommonBinaryHead(nn.Module):
    def __init__(self, in_features: int):
        super().__init__()
        self.in_features = int(in_features)
        self.net = nn.Sequential(
            nn.BatchNorm1d(self.in_features),
            nn.Linear(self.in_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.50),
            nn.Linear(512, 256),
            nn.Dropout(0.30),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        return self.net(x)

class TorchvisionBinaryModel(nn.Module):
    """Original torchvision backbone with classifier/fc removed; common head appended."""
    def __init__(self, model_id: str, pretrained=True):
        super().__init__()
        self.model_id = model_id
        if model_id == "vgg16":
            m = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1 if pretrained else None)
            in_features = m.classifier[0].in_features
            m.classifier = nn.Identity()
        elif model_id == "resnet50":
            m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
            in_features = m.fc.in_features
            m.fc = nn.Identity()
        elif model_id == "densenet201":
            m = models.densenet201(weights=models.DenseNet201_Weights.IMAGENET1K_V1 if pretrained else None)
            in_features = m.classifier.in_features
            m.classifier = nn.Identity()
        elif model_id == "efficientnet_b0":
            m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
            in_features = m.classifier[1].in_features
            m.classifier = nn.Identity()
        elif model_id == "mobilenet_v2":
            m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V2 if pretrained else None)
            in_features = m.classifier[1].in_features
            m.classifier = nn.Identity()
        elif model_id == "mobilenet_v3_small":
            m = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1 if pretrained else None)
            in_features = m.classifier[0].in_features
            m.classifier = nn.Identity()
        else:
            raise KeyError(model_id)
        self.backbone = m
        self.head = CommonBinaryHead(in_features)
        self.feature_dim = int(in_features)

    def forward(self, x):
        return self.head(self.backbone(x))

class TimmBinaryModel(nn.Module):
    """Original timm architecture with its classifier removed; common head appended."""
    def __init__(self, timm_name: str, pretrained=True):
        super().__init__()
        self.timm_name = timm_name
        self.backbone = timm.create_model(
            timm_name,
            pretrained=pretrained,
            num_classes=0,
            global_pool="avg",
        )
        feature_dim = int(self.backbone.num_features)
        self.head = CommonBinaryHead(feature_dim)
        self.feature_dim = feature_dim

    def forward(self, x):
        features = self.backbone(x)
        if features.ndim > 2:
            features = features.flatten(1)
        return self.head(features)

TIMM_NAMES = {
    "convnext_tiny": "convnext_tiny",
    "swin_tiny_patch4_window7_224": "swin_tiny_patch4_window7_224",
    "vit_base_patch16_224": "vit_base_patch16_224",
}

def build_model(model_id: str, pretrained=True):
    if model_id in {"vgg16", "resnet50", "densenet201", "efficientnet_b0", "mobilenet_v2", "mobilenet_v3_small"}:
        model = TorchvisionBinaryModel(model_id, pretrained=pretrained)
    elif model_id in TIMM_NAMES:
        model = TimmBinaryModel(TIMM_NAMES[model_id], pretrained=pretrained)
    else:
        raise KeyError(model_id)
    return model.to(DEVICE)


def set_train_stage(model, stage: int):
    if stage == 1:
        model.backbone.eval()
        model.head.train()
        for p in model.backbone.parameters():
            p.requires_grad = False
        for p in model.head.parameters():
            p.requires_grad = True
    elif stage == 2:
        model.train()
        for p in model.parameters():
            p.requires_grad = True
    else:
        raise ValueError(stage)

# Build each architecture with pretrained=False for a fast structural smoke test.
model_specs = []
for model_id in BENCHMARK_MODELS:
    m = build_model(model_id, pretrained=False)
    total = sum(p.numel() for p in m.parameters())
    model_specs.append({
        "model_id": model_id,
        "model": MODEL_DISPLAY_NAMES[model_id],
        "feature_dim": m.feature_dim,
        "parameters_M": total / 1e6,
        "common_head": COMMON_HEAD_DESCRIPTION,
        "library": "torchvision" if isinstance(m, TorchvisionBinaryModel) else "timm",
    })
    del m

pd.DataFrame(model_specs).to_csv(RESULT_DIR / "model_specs.csv", index=False)
print(pd.DataFrame(model_specs).to_string(index=False))

                    model_id             model  feature_dim  parameters_M                                                                                                                     common_head     library
          mobilenet_v3_small MobileNetV3-Small          576      1.355169 BatchNorm1d(in_features) -> Linear(in_features,512) -> ReLU -> Dropout(0.5) -> Linear(512,256) -> Dropout(0.3) -> Linear(256,1) torchvision
                mobilenet_v2       MobileNetV2         1280      3.013889 BatchNorm1d(in_features) -> Linear(in_features,512) -> ReLU -> Dropout(0.5) -> Linear(512,256) -> Dropout(0.3) -> Linear(256,1) torchvision
             efficientnet_b0   EfficientNet-B0         1280      4.797565 BatchNorm1d(in_features) -> Linear(in_features,512) -> ReLU -> Dropout(0.5) -> Linear(512,256) -> Dropout(0.3) -> Linear(256,1) torchvision
                 densenet201       DenseNet201         1920     19.211905 BatchNorm1d(in_features) -> Linear(in_features,512) -> ReLU -> Dropout

In [6]:
# ==============================================================================
# CELL 6 — SANITY CHECK: FORWARD PASS, BACKPROP, SPLIT, THRESHOLD, ONNX SMOKE TEST
# ===============================================================================

# 1. One batch must flow through MobileNetV3-Small end-to-end.
sanity_model = build_model("mobilenet_v3_small", pretrained=False)
set_train_stage(sanity_model, 1)
x0, y0 = next(iter(train_loader))
with torch.no_grad():
    out0 = sanity_model(x0.to(DEVICE))
assert out0.shape == (x0.shape[0], 1)
assert torch.isfinite(out0).all()

a_pos = float(np.sum(train["label"].values == 0)) / len(train)
a_neg = float(np.sum(train["label"].values == 1)) / len(train)
print("Sanity forward shape:", tuple(out0.shape))
print("Focal weights:", a_pos, a_neg)

# 2. Test the exact thresholding routine on a toy example.

def choose_threshold_by_youden(y, p):
    fpr, tpr, thresholds = roc_curve(y, p)
    finite = np.isfinite(thresholds)
    if not finite.any():
        raise RuntimeError("No finite ROC thresholds available.")
    idx = np.argmax(tpr[finite] - fpr[finite])
    return float(thresholds[finite][idx])

toy_tau = choose_threshold_by_youden(np.array([0,0,1,1]), np.array([0.1,0.2,0.8,0.9]))
assert 0.2 <= toy_tau <= 0.8
print("Threshold smoke test passed, tau=", toy_tau)

# 3. Model outputs must be identical before/after common-head training mode switch.
san = sanity_model.eval()
with torch.no_grad():
    out_eval = san(x0.to(DEVICE))
assert torch.isfinite(out_eval).all()

# 4. Split sizes / independence guard.
assert len(train) == 380 and len(val) == 55 and len(test) == 109
assert not (set(train.cleaned_path) & set(val.cleaned_path))
assert not (set(train.cleaned_path) & set(test.cleaned_path))
assert not (set(val.cleaned_path) & set(test.cleaned_path))

# 5. ONNX smoke test for a simple model. Full export/profiling occurs later.
smoke_path = MODEL_DIR / "mobilenet_v3_small_smoke.onnx"
try:
    cpu_sanity = deepcopy(sanity_model).to("cpu").eval()
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
    torch.onnx.export(
        cpu_sanity,
        dummy,
        str(smoke_path),
        input_names=["input"],
        output_names=["logit"],
        opset_version=ONNX_OPSET,
        do_constant_folding=True,
        dynamo=False,
    )
    sess = ort.InferenceSession(str(smoke_path), providers=["CPUExecutionProvider"])
    pred_onnx = sess.run(None, {sess.get_inputs()[0].name: dummy.numpy()})[0]
    with torch.no_grad():
        pred_pt = cpu_sanity(dummy).numpy()
    max_abs = float(np.max(np.abs(pred_onnx - pred_pt)))
    assert max_abs < 1e-3, f"ONNX mismatch too large: {max_abs}"
    print("ONNX smoke test PASSED. max_abs_error=", max_abs)
except Exception as exc:
    raise RuntimeError(
        "ONNX smoke test failed. Stop before the long benchmark and fix the environment/export path."
    ) from exc

print("\nALL SANITY CHECKS PASSED — safe to start the full benchmark.")

Sanity forward shape: (32, 1)
Focal weights: 0.2868421052631579 0.7131578947368421
Threshold smoke test passed, tau= 0.8


/tmp/ipykernel_105/1126788069.py:50: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX smoke test PASSED. max_abs_error= 5.587935447692871e-09

ALL SANITY CHECKS PASSED — safe to start the full benchmark.


In [7]:
# ==============================================================================
# CELL 7 — TRAINING CORE + STANDARDIZED NINE-MODEL BENCHMARK
# ==============================================================================

class BinaryFocalLoss(nn.Module):
    def __init__(self, alpha_pos, alpha_neg, gamma=2.0, eps=1e-7):
        super().__init__()
        self.alpha_pos = float(alpha_pos)
        self.alpha_neg = float(alpha_neg)
        self.gamma = float(gamma)
        self.eps = float(eps)

    def forward(self, logits, targets):
        p = torch.sigmoid(logits.float()).clamp(self.eps, 1.0 - self.eps)
        loss = (
            -self.alpha_pos * (1.0 - p).pow(self.gamma) * torch.log(p) * targets
            -self.alpha_neg * p.pow(self.gamma) * torch.log(1.0 - p) * (1.0 - targets)
        )
        return loss.mean()

alpha_pos = float(np.sum(train["label"].values == 0)) / len(train)
alpha_neg = float(np.sum(train["label"].values == 1)) / len(train)
FOCAL_CRITERION = BinaryFocalLoss(alpha_pos, alpha_neg, FOCAL_GAMMA).to(DEVICE)


def eval_loader(model, loader, criterion=None):
    model.eval()
    running_loss = 0.0
    seen = 0
    probs, labels = [], []
    with torch.inference_mode():
        for x, y in loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True).unsqueeze(1)
            logits = model(x)
            if criterion is not None:
                loss = criterion(logits, y)
                running_loss += float(loss.item()) * x.size(0)
                seen += x.size(0)
            probs.extend(torch.sigmoid(logits.float()).cpu().numpy().ravel())
            labels.extend(y.cpu().numpy().ravel())
    return {
        "loss": (running_loss / seen) if seen else np.nan,
        "probs": np.asarray(probs, dtype=float),
        "targets": np.asarray(labels, dtype=int),
    }


def train_stage(model, train_loader_local, val_loader_local, criterion, stage, lr, epochs, patience):
    set_train_stage(model, stage)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.AdamW(params, lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    best_loss = float("inf")
    best_state = None
    patience_counter = 0
    history = []

    for epoch in range(1, epochs + 1):
        if stage == 1:
            set_train_stage(model, 1)
        else:
            model.train()

        running_loss = 0.0
        seen = 0
        for x, y in train_loader_local:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True).unsqueeze(1)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logits = model(x)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += float(loss.item()) * x.size(0)
            seen += x.size(0)

        scheduler.step()
        train_loss = running_loss / max(seen, 1)
        val_out = eval_loader(model, val_loader_local, criterion)
        val_auc = roc_auc_score(val_out["targets"], val_out["probs"])
        history.append({
            "stage": stage,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_out["loss"],
            "val_auc": val_auc,
            "lr": optimizer.param_groups[0]["lr"],
        })

        if val_out["loss"] < best_loss - 1e-8:
            best_loss = val_out["loss"]
            best_state = deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    if best_state is None:
        raise RuntimeError(f"No checkpoint produced for stage {stage}.")
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)


def classification_metrics(y, p, threshold):
    pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y, pred)),
        "auc_roc": float(roc_auc_score(y, p)),
        "auc_pr": float(average_precision_score(y, p)),
        "f1": float(f1_score(y, pred, zero_division=0)),
        "sensitivity": float(tp / (tp + fn)) if (tp + fn) else np.nan,
        "specificity": float(tn / (tn + fp)) if (tn + fp) else np.nan,
        "mcc": float(matthews_corrcoef(y, pred)),
        "brier": float(brier_score_loss(y, p)),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

benchmark_results = {}
benchmark_predictions = {}
training_histories = {}

for model_id in BENCHMARK_MODELS:
    seed_everything(SEED)
    print(f"\n>>> FINAL BENCHMARK: {MODEL_DISPLAY_NAMES[model_id]}")
    local_train, local_val, local_test = make_loaders(train, val, test, full_aug=True, seed=SEED)
    model = build_model(model_id, pretrained=True)

    model, hist1 = train_stage(
        model, local_train, local_val, FOCAL_CRITERION,
        stage=1, lr=STAGE1_LR, epochs=STAGE1_EPOCHS, patience=STAGE1_PATIENCE,
    )
    model, hist2 = train_stage(
        model, local_train, local_val, FOCAL_CRITERION,
        stage=2, lr=STAGE2_LR, epochs=STAGE2_EPOCHS, patience=STAGE2_PATIENCE,
    )
    history = pd.concat([hist1, hist2], ignore_index=True)
    training_histories[model_id] = history
    history.to_csv(RESULT_DIR / f"{model_id}_training_history.csv", index=False)

    val_out = eval_loader(model, local_val, FOCAL_CRITERION)
    tau = choose_threshold_by_youden(val_out["targets"], val_out["probs"])
    test_out = eval_loader(model, local_test, criterion=None)
    metrics = classification_metrics(test_out["targets"], test_out["probs"], tau)

    checkpoint = MODEL_DIR / f"{model_id}_benchmark_seed{SEED}.pt"
    torch.save(model.state_dict(), checkpoint)
    pred_frame = pd.DataFrame({
        "path": test["cleaned_path"].values,
        "true_label": test_out["targets"],
        "probability": test_out["probs"],
        "prediction": (test_out["probs"] >= tau).astype(int),
    })
    pred_frame.to_csv(RESULT_DIR / f"{model_id}_test_predictions.csv", index=False)

    total_params = sum(p.numel() for p in model.parameters())
    benchmark_results[model_id] = {
        **metrics,
        "parameters_M": total_params / 1e6,
        "checkpoint": str(checkpoint),
        "feature_dim": int(model.feature_dim),
    }
    benchmark_predictions[model_id] = {
        "val_targets": val_out["targets"],
        "val_probs": val_out["probs"],
        "test_targets": test_out["targets"],
        "test_probs": test_out["probs"],
        "test_preds": (test_out["probs"] >= tau).astype(int),
    }

    print(
        f"Acc={metrics['accuracy']:.4f} | AUC={metrics['auc_roc']:.4f} | "
        f"Sens={metrics['sensitivity']:.4f} | Spec={metrics['specificity']:.4f} | tau={tau:.3f}"
    )

benchmark_df = pd.DataFrame.from_dict(benchmark_results, orient="index").reset_index(names="model_id")
benchmark_df["model"] = benchmark_df["model_id"].map(MODEL_DISPLAY_NAMES)
benchmark_df.to_csv(RESULT_DIR / "benchmark_master.csv", index=False)

print("\n=== STANDARDIZED NINE-MODEL BENCHMARK COMPLETE ===")
print(
    benchmark_df[
        ["model", "parameters_M", "accuracy", "auc_roc", "auc_pr", "sensitivity", "specificity", "mcc", "threshold"]
    ].to_string(index=False)
)


>>> FINAL BENCHMARK: MobileNetV3-Small
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 93.4MB/s]
/tmp/ipykernel_105/1506881063.py:54: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
/tmp/ipykernel_105/1506881063.py:73: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/tmp/ipykernel_105/1506881063.py:73: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/tmp/ipykernel_105/1506881063.py:73: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/tmp/ipykernel_105/1506881063.py:73: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. 

Acc=0.9541 | AUC=0.9806 | Sens=0.9872 | Spec=0.8710 | tau=0.587

>>> FINAL BENCHMARK: MobileNetV2
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 116MB/s] 
/tmp/ipykernel_105/1506881063.py:54: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
/tmp/ipykernel_105/1506881063.py:73: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/tmp/ipykernel_105/1506881063.py:73: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/tmp/ipykernel_105/1506881063.py:73: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/tmp/ipykernel_105/1506881063.py:73: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. 

Acc=0.8991 | AUC=0.9764 | Sens=0.8590 | Spec=1.0000 | tau=0.765

>>> FINAL BENCHMARK: EfficientNet-B0
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


...

/tmp/ipykernel_105/1506881063.py:54: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
/tmp/ipykernel_105/1506881063.py:73: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/tmp/ipykernel_105/1506881063.py:73: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/tmp/ipykernel_105/1506881063.py:73: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/tmp/ipykernel_105/1506881063.py:73: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` inst

Acc=0.9541 | AUC=0.9946 | Sens=0.9359 | Spec=1.0000 | tau=0.727

>>> FINAL BENCHMARK: ViT-B/16


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Unexpected keys (norm.bias, norm.weight) found while loading pretrained weights. This may be expected if model is being adapted.
/tmp/ipykernel_105/1506881063.py:54: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
/tmp/ipykernel_105/1506881063.py:73: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/tmp/ipykernel_105/1506881063.py:73: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/tmp/ipykernel_105/1506881063.py:73: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/tmp/ipykernel_105/1506881

Acc=0.9817 | AUC=1.0000 | Sens=0.9744 | Spec=1.0000 | tau=0.645

=== STANDARDIZED NINE-MODEL BENCHMARK COMPLETE ===
            model  parameters_M  accuracy  auc_roc   auc_pr  sensitivity  specificity      mcc  threshold
MobileNetV3-Small      1.355169  0.954128 0.980562 0.992157     0.987179     0.870968 0.886071   0.586779
      MobileNetV2      3.013889  0.899083 0.976427 0.991442     0.858974     1.000000 0.796244   0.764885
  EfficientNet-B0      4.797565  0.908257 0.949959 0.979548     0.948718     0.806452 0.770954   0.435842
      DenseNet201     19.211905  0.954128 0.997932 0.999228     0.935897     1.000000 0.897726   0.649420
         ResNet50     24.692801  0.926606 0.989661 0.995933     1.000000     0.741935 0.820316   0.262292
            VGG16     27.742017  0.926606 0.984698 0.993896     0.974359     0.806452 0.815990   0.305958
    ConvNeXt-Tiny     28.346977  0.972477 1.000000 1.000000     0.961538     1.000000 0.936321   0.777388
           Swin-T     28.046203  0.9

In [8]:
# ==============================================================================
# CELL 8 — BOOTSTRAP CIs + EXACT McNEMAR / HOLM
# ==============================================================================

BOOTSTRAP_RESAMPLES = 2000
BOOTSTRAP_SEED = 2026


def bootstrap_ci(y, pred, n_boot=BOOTSTRAP_RESAMPLES, seed=BOOTSTRAP_SEED):
    rng = np.random.default_rng(seed)
    y = np.asarray(y)
    pred = np.asarray(pred)
    accs, senss, specs = [], [], []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y), len(y))
        yy, pp = y[idx], pred[idx]
        tn, fp, fn, tp = confusion_matrix(yy, pp, labels=[0, 1]).ravel()
        accs.append((tp + tn) / len(yy))
        senss.append(tp / (tp + fn) if tp + fn else np.nan)
        specs.append(tn / (tn + fp) if tn + fp else np.nan)

    def interval(values):
        values = np.asarray(values, dtype=float)
        values = values[np.isfinite(values)]
        return float(np.percentile(values, 2.5)), float(np.percentile(values, 97.5))

    return interval(accs), interval(senss), interval(specs)


ci_rows = []
for model_id in BENCHMARK_MODELS:
    p = benchmark_predictions[model_id]
    ci_acc, ci_sens, ci_spec = bootstrap_ci(p["test_targets"], p["test_preds"])
    ci_rows.append({
        "model_id": model_id,
        "accuracy_lo": ci_acc[0], "accuracy_hi": ci_acc[1],
        "sensitivity_lo": ci_sens[0], "sensitivity_hi": ci_sens[1],
        "specificity_lo": ci_spec[0], "specificity_hi": ci_spec[1],
    })

def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    m = len(p_values)
    order = np.argsort(p_values)
    sorted_p = p_values[order]
    adjusted_sorted = np.maximum.accumulate((m - np.arange(m)) * sorted_p)
    adjusted = np.empty(m, dtype=float)
    adjusted[order] = np.minimum(adjusted_sorted, 1.0)
    return adjusted


ref = benchmark_predictions[REFERENCE_MODEL]
y = ref["test_targets"]
ref_pred = ref["test_preds"]
comparison_rows = []

for model_id in BENCHMARK_MODELS:
    if model_id == REFERENCE_MODEL:
        continue
    other = benchmark_predictions[model_id]["test_preds"]
    b = int(np.sum((ref_pred == y) & (other != y)))
    c = int(np.sum((ref_pred != y) & (other == y)))
    n = b + c
    if n == 0:
        p_raw = 1.0
    else:
        k = min(b, c)
        # Two-sided exact binomial test for McNemar's test.
        from math import comb
        tail = sum(comb(n, i) for i in range(k + 1)) / (2 ** n)
        p_raw = min(1.0, 2.0 * tail)
    comparison_rows.append({
        "reference": REFERENCE_MODEL,
        "model_id": model_id,
        "b_ref_correct_other_wrong": b,
        "c_ref_wrong_other_correct": c,
        "p_raw": float(p_raw),
    })

mcnemar_df = pd.DataFrame(comparison_rows)
mcnemar_df["p_holm"] = holm_adjust(mcnemar_df["p_raw"].values)
mcnemar_df.to_csv(RESULT_DIR / "mcnemar_holm.csv", index=False)

ci_df = pd.DataFrame(ci_rows)
final_benchmark_df = benchmark_df.merge(ci_df, on="model_id", how="left")
final_benchmark_df.to_csv(RESULT_DIR / "benchmark_master_with_ci.csv", index=False)

print("\n=== BENCHMARK WITH 95% CIs ===")
print(
    final_benchmark_df[
        ["model", "accuracy", "accuracy_lo", "accuracy_hi", "auc_roc",
         "sensitivity", "sensitivity_lo", "sensitivity_hi",
         "specificity", "specificity_lo", "specificity_hi", "mcc", "threshold"]
    ].to_string(index=False)
)
print("\n=== MCNEMAR / HOLM ===")
print(mcnemar_df.to_string(index=False))


=== BENCHMARK WITH 95% CIs ===
            model  accuracy  accuracy_lo  accuracy_hi  auc_roc  sensitivity  sensitivity_lo  sensitivity_hi  specificity  specificity_lo  specificity_hi      mcc  threshold
MobileNetV3-Small  0.954128     0.908257     0.990826 0.980562     0.987179        0.958904        1.000000     0.870968        0.733333        0.971429 0.886071   0.586779
      MobileNetV2  0.899083     0.844037     0.954128 0.976427     0.858974        0.774991        0.931507     1.000000        1.000000        1.000000 0.796244   0.764885
  EfficientNet-B0  0.908257     0.852982     0.954128 0.949959     0.948718        0.894721        0.987952     0.806452        0.653846        0.933333 0.770954   0.435842
      DenseNet201  0.954128     0.908257     0.990826 0.997932     0.935897        0.879518        0.986671     1.000000        1.000000        1.000000 0.897726   0.649420
         ResNet50  0.926606     0.871560     0.972477 0.989661     1.000000        1.000000        1.00

In [9]:
# ==============================================================================
# CELL 9 — RUNTIME PROFILING: REAL MEASUREMENTS ONLY
# ==============================================================================


def profile_gpu(model):
    if DEVICE.type != "cuda":
        return np.nan
    model = model.eval()
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    with torch.inference_mode():
        for _ in range(GPU_WARMUPS):
            _ = model(dummy)
        torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(GPU_ITERS):
            _ = model(dummy)
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
    return 1000.0 * elapsed / GPU_ITERS


def profile_cpu(model):
    model = deepcopy(model).to("cpu").eval()
    torch.set_num_threads(CPU_THREADS)
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
    with torch.inference_mode():
        for _ in range(CPU_WARMUPS):
            _ = model(dummy)
        start = time.perf_counter()
        for _ in range(CPU_ITERS):
            _ = model(dummy)
        elapsed = time.perf_counter() - start
    return 1000.0 * elapsed / CPU_ITERS


def profile_onnx(model, model_id):
    model = deepcopy(model).to("cpu").eval()
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
    path = MODEL_DIR / f"{model_id}_benchmark.onnx"

    torch.onnx.export(
        model,
        dummy,
        str(path),
        input_names=["input"],
        output_names=["logit"],
        opset_version=ONNX_OPSET,
        do_constant_folding=True,
        dynamo=False,
    )

    sess_options = ort.SessionOptions()
    sess_options.intra_op_num_threads = CPU_THREADS
    sess_options.inter_op_num_threads = 1
    sess = ort.InferenceSession(
        str(path),
        sess_options=sess_options,
        providers=["CPUExecutionProvider"],
    )
    input_name = sess.get_inputs()[0].name
    data = dummy.numpy()

    # Accuracy guard: compare one deterministic ONNX prediction with PyTorch.
    with torch.inference_mode():
        pt_out = model(dummy).numpy()
    onnx_out = sess.run(None, {input_name: data})[0]
    max_abs = float(np.max(np.abs(pt_out - onnx_out)))
    if max_abs >= 1e-3:
        raise RuntimeError(f"ONNX numerical mismatch for {model_id}: {max_abs}")

    for _ in range(ONNX_WARMUPS):
        _ = sess.run(None, {input_name: data})
    start = time.perf_counter()
    for _ in range(ONNX_ITERS):
        _ = sess.run(None, {input_name: data})
    elapsed = time.perf_counter() - start

    return {
        "onnx_ms": 1000.0 * elapsed / ONNX_ITERS,
        "onnx_size_MB": path.stat().st_size / (1024 ** 2),
        "onnx_max_abs_error": max_abs,
        "onnx_path": str(path),
    }

profile_rows = []
for model_id in BENCHMARK_MODELS:
    model = build_model(model_id, pretrained=False)
    ckpt = MODEL_DIR / f"{model_id}_benchmark_seed{SEED}.pt"
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    model.eval()

    total_params = sum(p.numel() for p in model.parameters())
    row = {
        "model_id": model_id,
        "model": MODEL_DISPLAY_NAMES[model_id],
        "parameters_M": total_params / 1e6,
        "gpu_ms": profile_gpu(model) if PROFILE_GPU else np.nan,
        "pytorch_cpu_ms": profile_cpu(model) if PROFILE_PYTORCH_CPU else np.nan,
    }

    if PROFILE_ONNX:
        try:
            row.update(profile_onnx(model, model_id))
        except Exception as exc:
            if STRICT_ONNX:
                raise RuntimeError(
                    f"ONNX profiling failed for {model_id}. No fallback value is permitted."
                ) from exc
            row.update({
                "onnx_ms": np.nan,
                "onnx_size_MB": np.nan,
                "onnx_max_abs_error": np.nan,
                "onnx_path": "FAILED",
            })
    else:
        row.update({
            "onnx_ms": np.nan,
            "onnx_size_MB": np.nan,
            "onnx_max_abs_error": np.nan,
            "onnx_path": "DISABLED",
        })

    row["gpu_fps"] = 1000.0 / row["gpu_ms"] if np.isfinite(row["gpu_ms"]) else np.nan
    row["onnx_fps_proxy"] = 1000.0 / row["onnx_ms"] if np.isfinite(row["onnx_ms"]) else np.nan
    profile_rows.append(row)

df_profile = pd.DataFrame(profile_rows)
df_profile.to_csv(RESULT_DIR / "runtime_profile.csv", index=False)

print(df_profile[
    ["model", "parameters_M", "gpu_ms", "pytorch_cpu_ms", "onnx_ms", "onnx_fps_proxy", "onnx_size_MB"]
].to_string(index=False))
print("\nONNX CPU = edge-oriented computational proxy; no physical smartphone/ARM claim is made.")

/tmp/ipykernel_105/4144271095.py:42: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.12/dist-packages/torch/__init__.py:2228: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert condition, message


            model  parameters_M    gpu_ms  pytorch_cpu_ms    onnx_ms  onnx_fps_proxy  onnx_size_MB
MobileNetV3-Small      1.355169  5.609268       11.617665   2.039218      490.383953      5.187465
      MobileNetV2      3.013889  5.548179       22.106761   5.262020      190.041092     11.486239
  EfficientNet-B0      4.797565  7.952748       33.081411   9.579646      104.387992     18.310276
      DenseNet201     19.211905 24.017985      147.789619  57.191483       17.485121     74.356904
         ResNet50     24.692801  6.294935       80.198387  40.193185       24.879839     94.137502
            VGG16     27.742017  8.652351      172.190725 136.026501        7.351509    106.027800
    ConvNeXt-Tiny     28.346977  9.469562       73.838040  61.084342       16.370807    108.227518
           Swin-T     28.046203 17.191047       99.680916  82.465392       12.126299    109.666063
         ViT-B/16     86.325505 17.700540      207.702879 180.706289        5.533842    329.426627

ONNX CPU 

In [10]:
# ==============================================================================
# CELL 10 — PUBLICATION FIGURES AND TABLE EXPORT
# ==============================================================================

import matplotlib.pyplot as plt

# Figure 1: dataset samples + class labels.
fig, axes = plt.subplots(2, 4, figsize=(8.5, 4.4), dpi=300)
for idx, (_, row) in enumerate(train[train["label"] == 1].head(4).iterrows()):
    img = cv2.cvtColor(cv2.imread(row["cleaned_path"]), cv2.COLOR_BGR2RGB)
    axes[0, idx].imshow(cv2.resize(img, (224, 224)))
    axes[0, idx].set_title(f"Conjunctivitis #{idx+1}", fontsize=8, fontweight="bold")
    axes[0, idx].axis("off")
for idx, (_, row) in enumerate(train[train["label"] == 0].head(4).iterrows()):
    img = cv2.cvtColor(cv2.imread(row["cleaned_path"]), cv2.COLOR_BGR2RGB)
    axes[1, idx].imshow(cv2.resize(img, (224, 224)))
    axes[1, idx].set_title(f"Non-Conjunctivitis #{idx+1}", fontsize=8, fontweight="bold")
    axes[1, idx].axis("off")
plt.tight_layout()
plt.savefig(PLOT_DIR / "figure1_dataset_samples.png", dpi=300, bbox_inches="tight")
plt.close()

# Figure 2: MobileNetV3-Small and ConvNeXt-Tiny loss curves.
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.7), dpi=300)
for ax, model_id, title in [
    (axes[0], "mobilenet_v3_small", "(a) MobileNetV3-Small"),
    (axes[1], "convnext_tiny", "(b) ConvNeXt-Tiny"),
]:
    h = training_histories[model_id]
    ax.plot(h["train_loss"], label="Train Loss", lw=1.5)
    ax.plot(h["val_loss"], label="Val Loss", lw=1.5, ls="--")
    ax.set_title(title, fontsize=8.5, fontweight="bold")
    ax.set_xlabel("Epoch", fontsize=7.5)
    ax.set_ylabel("Focal Loss", fontsize=7.5)
    ax.legend(fontsize=7)
    ax.grid(True, ls="--", alpha=0.4)
plt.tight_layout()
plt.savefig(PLOT_DIR / "figure2_training_curves.png", dpi=300, bbox_inches="tight")
plt.close()

# Figure 3: ROC and PR curves for all nine models.
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.2), dpi=300)
for model_id in BENCHMARK_MODELS:
    p = benchmark_predictions[model_id]
    fpr, tpr, _ = roc_curve(p["test_targets"], p["test_probs"])
    axes[0].plot(fpr, tpr, lw=1.1, label=f"{MODEL_DISPLAY_NAMES[model_id]} ({roc_auc_score(p['test_targets'], p['test_probs']):.3f})")
    # PR curve via threshold-free precision/recall reconstruction.
    from sklearn.metrics import precision_recall_curve
    prec, rec, _ = precision_recall_curve(p["test_targets"], p["test_probs"])
    axes[1].plot(rec, prec, lw=1.1, label=f"{MODEL_DISPLAY_NAMES[model_id]} ({average_precision_score(p['test_targets'], p['test_probs']):.3f})")
axes[0].plot([0,1], [0,1], "k--", alpha=0.3)
axes[0].set_title("ROC Curves", fontsize=8.5, fontweight="bold")
axes[0].set_xlabel("False Positive Rate", fontsize=7.5)
axes[0].set_ylabel("True Positive Rate", fontsize=7.5)
axes[1].set_title("Precision–Recall Curves", fontsize=8.5, fontweight="bold")
axes[1].set_xlabel("Recall", fontsize=7.5)
axes[1].set_ylabel("Precision", fontsize=7.5)
axes[0].legend(fontsize=5.4, loc="lower right")
axes[1].legend(fontsize=5.4, loc="lower left")
plt.tight_layout()
plt.savefig(PLOT_DIR / "figure3_roc_pr_curves.png", dpi=300, bbox_inches="tight")
plt.close()

# Figure 4: predictive / compute trade-off.
merged = final_benchmark_df.merge(
    df_profile[["model_id", "onnx_ms", "onnx_size_MB"]],
    on="model_id", how="left"
)
fig, ax = plt.subplots(figsize=(5.8, 3.8), dpi=300)
for _, r in merged.iterrows():
    ax.scatter(r["onnx_ms"], r["accuracy"], s=28)
    ax.annotate(r["model"], (r["onnx_ms"], r["accuracy"]), fontsize=6, xytext=(3,3), textcoords="offset points")
ax.set_xlabel("ONNX Runtime CPU latency (ms)", fontsize=8)
ax.set_ylabel("Test accuracy", fontsize=8)
ax.set_title("Predictive performance vs. edge-oriented CPU latency", fontsize=9, fontweight="bold")
ax.grid(True, ls="--", alpha=0.35)
plt.tight_layout()
plt.savefig(PLOT_DIR / "figure4_compute_tradeoff.png", dpi=300, bbox_inches="tight")
plt.close()

# Figure 5: compact confusion matrices for the three representative models.
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.4), dpi=300)
for ax, model_id in zip(axes, [PRIMARY_MODEL, SECONDARY_MODEL, HIGH_CAPACITY_REFERENCE]):
    p = benchmark_predictions[model_id]
    cm = confusion_matrix(p["test_targets"], p["test_preds"], labels=[0,1])
    ax.imshow(cm)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, int(cm[i,j]), ha="center", va="center", fontsize=10, fontweight="bold")
    ax.set_xticks([0,1], ["Non-Conj", "Conj"])
    ax.set_yticks([0,1], ["Non-Conj", "Conj"])
    tau = benchmark_results[model_id]["threshold"]
    ax.set_title(f"{MODEL_DISPLAY_NAMES[model_id]}\n(tau*={tau:.2f})", fontsize=7.5, fontweight="bold")
    ax.set_xlabel("Predicted", fontsize=7.5)
axes[0].set_ylabel("True", fontsize=7.5)
plt.tight_layout()
plt.savefig(PLOT_DIR / "figure5_compact_confusion_matrices.png", dpi=300, bbox_inches="tight")
plt.close()

# Publication-ready CSVs.
merged.to_csv(TABLE_DIR / "table2_benchmark_runtime.csv", index=False)
mcnemar_df.to_csv(TABLE_DIR / "table3_mcnemar_holm.csv", index=False)

# IEEE-friendly LaTeX row generator. Do not manually edit numerical values after this cell.
def latex_num(v, digits=4):
    return f"{float(v):.{digits}f}"

latex_rows = []
for _, r in merged.iterrows():
    latex_rows.append(
        f"{r['model']} & {r['parameters_M']:.2f} & {r['onnx_ms']:.1f} & "
        f"{r['accuracy']:.4f} & {r['auc_roc']:.4f} & {r['sensitivity']:.4f} & "
        f"{r['specificity']:.4f} & {r['mcc']:.4f} & {r['threshold']:.2f} \\\\"
    )
(TABLE_DIR / "table2_latex_rows.txt").write_text("\n".join(latex_rows))

registry = {
    "paper_title": PAPER_TITLE,
    "primary_model": PRIMARY_MODEL,
    "secondary_model": SECONDARY_MODEL,
    "high_capacity_reference": HIGH_CAPACITY_REFERENCE,
    "reference_model": REFERENCE_MODEL,
    "cohort": {"n": int(len(df_clean_manifest)), "conjunctivitis": 388, "non_conjunctivitis": 156},
    "split": {"train": len(train), "validation": len(val), "test": len(test)},
    "test_used_for_tuning": False,
    "test_used_for_threshold_selection": False,
    "common_head": COMMON_HEAD_DESCRIPTION,
    "onnx_fallback": False,
    "physical_edge_device_test": False,
}
(RESULT_DIR / "final_registry.json").write_text(json.dumps(registry, indent=2))
print("Publication figures/tables exported to:", PLOT_DIR, TABLE_DIR)

Publication figures/tables exported to: /kaggle/working/plots /kaggle/working/tables


In [11]:
# ==============================================================================
# CELL 11 — OPTIONAL: MOBILEV3-SMALL VALIDATION-ONLY OPTIMIZATION STUDY
# ==============================================================================

if RUN_MOBILENET_OPTIMIZATION:
    print("Running separate MobileNetV3-Small validation-only optimization study.")

    @dataclass(frozen=True)
    class MobileCfg:
        name: str
        use_stage2: bool
        stage2_lr: float
        use_focal: bool
        full_aug: bool
        dropout1: float
        dropout2: float

    configs = [
        MobileCfg("baseline", True, 1e-5, True, True, 0.50, 0.30),
        MobileCfg("no_stage2", False, 0.0, True, True, 0.50, 0.30),
        MobileCfg("bce", True, 1e-5, False, True, 0.50, 0.30),
        MobileCfg("mild_aug", True, 1e-5, True, False, 0.50, 0.30),
        MobileCfg("stage2_3e5", True, 3e-5, True, True, 0.50, 0.30),
    ]

    def build_mob_with_cfg(cfg):
        m = build_model("mobilenet_v3_small", pretrained=True)
        m.head = CommonBinaryHead(m.feature_dim)
        m.head.net[3] = nn.Dropout(cfg.dropout1)
        m.head.net[5] = nn.Dropout(cfg.dropout2)
        return m.to(DEVICE)

    opt_rows = []
    for cfg in configs:
        for seed in MOBILENET_OPTIMIZATION_SEEDS:
            seed_everything(seed)
            tr_loader, va_loader, _ = make_loaders(train, val, test, full_aug=cfg.full_aug, seed=seed)
            m = build_mob_with_cfg(cfg)
            criterion = FOCAL_CRITERION if cfg.use_focal else nn.BCEWithLogitsLoss()
            m, _ = train_stage(m, tr_loader, va_loader, criterion, 1, STAGE1_LR, STAGE1_EPOCHS, STAGE1_PATIENCE)
            if cfg.use_stage2:
                m, _ = train_stage(m, tr_loader, va_loader, criterion, 2, cfg.stage2_lr, STAGE2_EPOCHS, STAGE2_PATIENCE)
            val_out = eval_loader(m, va_loader, criterion)
            tau = choose_threshold_by_youden(val_out["targets"], val_out["probs"])
            vm = classification_metrics(val_out["targets"], val_out["probs"], tau)
            opt_rows.append({
                "config": cfg.name,
                "seed": seed,
                "val_auc": vm["auc_roc"],
                "val_balanced_accuracy": vm["balanced_accuracy"],
                "val_sensitivity": vm["sensitivity"],
                "val_specificity": vm["specificity"],
                "val_mcc": vm["mcc"],
                "val_threshold": tau,
            })

    mobilenet_opt_df = pd.DataFrame(opt_rows)
    mobilenet_summary = (
        mobilenet_opt_df.groupby("config", as_index=False)
        .agg(
            mean_val_auc=("val_auc", "mean"),
            sd_val_auc=("val_auc", "std"),
            mean_val_balanced_accuracy=("val_balanced_accuracy", "mean"),
            mean_val_sensitivity=("val_sensitivity", "mean"),
            mean_val_specificity=("val_specificity", "mean"),
            mean_val_mcc=("val_mcc", "mean"),
        )
        .sort_values(["mean_val_auc", "mean_val_balanced_accuracy", "mean_val_mcc"], ascending=False)
    )
    mobilenet_opt_df.to_csv(RESULT_DIR / "mobilenet_validation_optimization.csv", index=False)
    mobilenet_summary.to_csv(RESULT_DIR / "mobilenet_validation_optimization_summary.csv", index=False)
    print(mobilenet_summary.to_string(index=False))
else:
    print("MobileNet optimization disabled. The standardized benchmark remains the primary final experiment.")

MobileNet optimization disabled. The standardized benchmark remains the primary final experiment.


In [12]:
# ==============================================================================
# CELL 11 — OPTIONAL QUALITATIVE XAI (MODEL-AGNOSTIC INPUT-GRADIENT SALIENCY)
# ==============================================================================

# Post-hoc qualitative analysis only. This cell does not affect model selection,
# thresholds, hyperparameters, benchmark tables, or the final test decision.
# It deliberately uses input-gradient saliency because it is model-agnostic and
# robust to differences in timm/torchvision internal attention implementations.
# When this cell is used, the manuscript should call the method "input-gradient
# saliency", not Grad-CAM or attention rollout.

if RUN_XAI:
    import matplotlib.pyplot as plt

    def saliency_map(model, rgb_uint8):
        model.eval()
        x = eval_transform(image=rgb_uint8)["image"].unsqueeze(0).to(DEVICE)
        x.requires_grad_(True)
        model.zero_grad(set_to_none=True)
        logit = model(x).sum()
        logit.backward()
        grad = x.grad.detach().abs().mean(dim=1).squeeze().cpu().numpy()
        grad -= grad.min()
        if grad.max() > 0:
            grad /= grad.max()
        return grad

    rows = test.reset_index(drop=True).iloc[:4]
    representative_models = [PRIMARY_MODEL, SECONDARY_MODEL, HIGH_CAPACITY_REFERENCE]
    fig, axes = plt.subplots(4, 4, figsize=(8.6, 7.6), dpi=300)

    for r_idx, (_, row) in enumerate(rows.iterrows()):
        bgr = cv2.imread(row["cleaned_path"])
        if bgr is None:
            raise RuntimeError(f"XAI image read failure: {row['cleaned_path']}")
        rgb = cv2.resize(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), (IMG_SIZE, IMG_SIZE))
        axes[r_idx, 0].imshow(rgb)
        axes[r_idx, 0].set_title(f"Case {r_idx+1}", fontsize=8, fontweight="bold")
        axes[r_idx, 0].axis("off")

        for c_idx, model_id in enumerate(representative_models, start=1):
            model = build_model(model_id, pretrained=False)
            checkpoint = MODEL_DIR / f"{model_id}_benchmark_seed{SEED}.pt"
            state = torch.load(checkpoint, map_location=DEVICE)
            try:
                model.load_state_dict(state)
            except TypeError:
                model.load_state_dict(state)
            sal = saliency_map(model, rgb)
            axes[r_idx, c_idx].imshow(rgb)
            axes[r_idx, c_idx].imshow(sal, alpha=0.45, cmap="jet")
            axes[r_idx, c_idx].set_title(MODEL_DISPLAY_NAMES[model_id], fontsize=8)
            axes[r_idx, c_idx].axis("off")
            del model
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

    plt.tight_layout()
    out_path = PLOT_DIR / "figure6_xai_input_gradient_saliency.png"
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()
    print("Saved:", out_path)
else:
    print("XAI disabled. The core benchmark does not depend on this cell.")

XAI disabled. The core benchmark does not depend on this cell.


In [13]:
# ==============================================================================
# CELL 12 — FINAL INTEGRITY GATE
# ==============================================================================

assert len(BENCHMARK_MODELS) == 9
assert REFERENCE_MODEL == PRIMARY_MODEL
assert DROP_LAST is False
assert len(df_clean_manifest) == 544
assert df_clean_manifest["class"].value_counts().to_dict() == {
    "Conjunctivitis": 388,
    "Non_Conjunctivitis": 156,
}
assert (len(train), len(val), len(test)) == (380, 55, 109)
assert all(final_benchmark_df["threshold"].notna())
assert all(final_benchmark_df["accuracy"].between(0,1))
assert all(final_benchmark_df["auc_roc"].between(0,1))
assert all(final_benchmark_df["sensitivity"].between(0,1))
assert all(final_benchmark_df["specificity"].between(0,1))
assert not set(train.cleaned_path) & set(val.cleaned_path)
assert not set(train.cleaned_path) & set(test.cleaned_path)
assert not set(val.cleaned_path) & set(test.cleaned_path)

if PROFILE_ONNX and STRICT_ONNX:
    assert df_profile["onnx_ms"].notna().all()
    assert (df_profile["onnx_ms"] > 0).all()
    assert (df_profile["onnx_max_abs_error"] < 1e-3).all()

# Verify test predictions are exactly reproduced by their validation-locked thresholds.
for model_id in BENCHMARK_MODELS:
    tau = float(benchmark_results[model_id]["threshold"])
    probs = benchmark_predictions[model_id]["test_probs"]
    preds = benchmark_predictions[model_id]["test_preds"]
    assert np.array_equal(preds, (probs >= tau).astype(int))

print("\n" + "="*88)
print("FINAL INTEGRITY GATE: PASSED")
print("="*88)
print("• 544-image cohort reproduced and frozen")
print("• 380/55/109 train/validation/test split verified")
print("• All nine backbone identities preserved")
print("• Common classifier-head topology applied explicitly")
print("• drop_last=False")
print("• Validation-only threshold selection")
print("• No test-set tuning in the standardized benchmark")
print("• Exact paired McNemar + Holm correction")
print("• Bootstrap confidence intervals generated")
print("• ONNX measurements are real; no fabricated fallback")
print("• Physical mobile deployment is not claimed")
print("• Paper outputs are exported from one canonical result registry")


FINAL INTEGRITY GATE: PASSED
• 544-image cohort reproduced and frozen
• 380/55/109 train/validation/test split verified
• All nine backbone identities preserved
• Common classifier-head topology applied explicitly
• drop_last=False
• Validation-only threshold selection
• No test-set tuning in the standardized benchmark
• Exact paired McNemar + Holm correction
• Bootstrap confidence intervals generated
• ONNX measurements are real; no fabricated fallback
• Physical mobile deployment is not claimed
• Paper outputs are exported from one canonical result registry


In [14]:
# ==============================================================================
# CELL 13 — REVISED PUBLICATION FIGURES: BALANCED BINARY XAI & CLEAN COMPUTE TRADEOFF
# ==============================================================================

import subprocess
import sys
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

try:
  from pytorch_grad_cam import GradCAM
  from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
except ImportError:
  subprocess.check_call(
      [sys.executable, "-m", "pip", "install", "-q", "grad-cam"]
  )
  from pytorch_grad_cam import GradCAM
  from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# ------------------------------------------------------------------------------
# 1. GENERATE CLEAN, NON-OVERLAPPING FIGURE 4 (COMPUTE TRADEOFF)
# ------------------------------------------------------------------------------
print(">>> Re-generating Figure 4 (Predictive vs. Latency Trade-off)...")

merged_perf = final_benchmark_df.merge(
    df_profile[["model_id", "onnx_ms"]], on="model_id"
)

colors = {
    "MobileNetV3-Small": "#1f77b4",
    "MobileNetV2": "#ff7f0e",
    "EfficientNet-B0": "#2ca02c",
    "DenseNet201": "#d62728",
    "ResNet50": "#9467bd",
    "VGG16": "#8c564b",
    "ConvNeXt-Tiny": "#e377c2",
    "Swin-T": "#7f7f7f",
    "ViT-B/16": "#bcbd22",
}

# Dedicated offsets to prevent any text collisions
offsets = {
    "MobileNetV3-Small": (6, 4),
    "MobileNetV2": (6, 5),  # Positioned above dot to avoid axis clipping
    "EfficientNet-B0": (6, 4),
    "DenseNet201": (-74, 6),  # Shifted left-up away from ConvNeXt-Tiny
    "ConvNeXt-Tiny": (6, 7),  # Shifted right-up
    "ResNet50": (6, 4),
    "VGG16": (6, 4),
    "Swin-T": (6, -13),  # Shifted down-right to avoid ConvNeXt-Tiny
    "ViT-B/16": (-46, -13),
}

fig, ax = plt.subplots(figsize=(6.2, 4.0), dpi=300)
for _, r in merged_perf.iterrows():
  ax.scatter(r["onnx_ms"], r["accuracy"], color=colors[r["model"]], s=42, zorder=4)
  off = offsets.get(r["model"], (6, 4))
  ax.annotate(
      r["model"],
      (r["onnx_ms"], r["accuracy"]),
      xytext=off,
      textcoords="offset points",
      fontsize=7.2,
      fontweight="medium",
  )

ax.set_xlabel("ONNX Runtime CPU Latency (ms)", fontsize=8.5, labelpad=5)
ax.set_ylabel("Held-out Test Accuracy", fontsize=8.5, labelpad=5)
ax.set_title(
    "Predictive Performance vs. Edge-Oriented CPU Latency",
    fontsize=9.5,
    fontweight="bold",
    pad=8,
)
ax.set_ylim(0.865, 1.005)
ax.set_xlim(-5, 185)
ax.grid(True, linestyle="--", alpha=0.35, zorder=1)
plt.tight_layout()

fig4_path = PLOT_DIR / "figure4_compute_tradeoff.png"
plt.savefig(fig4_path, dpi=300, bbox_inches="tight")
plt.close()
print("Saved non-overlapping Figure 4 to:", fig4_path)

# ------------------------------------------------------------------------------
# 2. GENERATE BALANCED BINARY XAI GRAD-CAM (2 CONJUNCTIVITIS, 2 NON-CONJUNCTIVITIS)
# ------------------------------------------------------------------------------
print(
    "\n>>> Generating Balanced Binary Explainability (Grad-CAM) across both"
    " classes..."
)

test_df = pd.read_csv(SPLIT_DIR / "test_split.csv").reset_index(drop=True)
ref_preds = pd.read_csv(RESULT_DIR / "mobilenet_v3_small_test_predictions.csv")

probs = ref_preds["probability"].values
targets = ref_preds["true_label"].values
tau_ref = benchmark_results["mobilenet_v3_small"]["threshold"]

pos_indices = np.where(targets == 1)[0]
neg_indices = np.where(targets == 0)[0]

# Pick representative instances across both classes
idx_pos_high = pos_indices[np.argmax(probs[pos_indices])]
idx_pos_near = pos_indices[np.argmin(np.abs(probs[pos_indices] - tau_ref))]
idx_neg_high = neg_indices[np.argmin(probs[neg_indices])]
idx_neg_near = neg_indices[np.argmin(np.abs(probs[neg_indices] - tau_ref))]

selected_cases = [
    (
        "Case 1: Conjunctivitis (High-Conf)",
        idx_pos_high,
        test_df.loc[idx_pos_high, "cleaned_path"],
        1,
    ),
    (
        "Case 2: Conjunctivitis (Near-Thresh)",
        idx_pos_near,
        test_df.loc[idx_pos_near, "cleaned_path"],
        1,
    ),
    (
        "Case 3: Non-Conjunctivitis (High-Conf Control)",
        idx_neg_high,
        test_df.loc[idx_neg_high, "cleaned_path"],
        0,
    ),
    (
        "Case 4: Non-Conjunctivitis (Near-Thresh Control)",
        idx_neg_near,
        test_df.loc[idx_neg_near, "cleaned_path"],
        0,
    ),
]

pd.DataFrame([
    {
        "case": c[0],
        "test_idx": c[1],
        "path": c[2],
        "label": "Conjunctivitis" if c[3] == 1 else "Non_Conjunctivitis",
    }
    for c in selected_cases
]).to_csv(RESULT_DIR / "xai_balanced_selected_cases.csv", index=False)

XAI_MODELS = ["mobilenet_v3_small", "convnext_tiny", "vit_base_patch16_224"]
XAI_NAMES = {
    "mobilenet_v3_small": "MobileNetV3-Small",
    "convnext_tiny": "ConvNeXt-Tiny",
    "vit_base_patch16_224": "ViT-B/16",
}


def load_xai_model(mid):
  m = build_model(mid, pretrained=False)
  m.load_state_dict(
      torch.load(
          MODEL_DIR / f"{mid}_benchmark_seed{SEED}.pt", map_location=DEVICE
      ),
      strict=True,
  )
  return m.eval()


def get_target_layer_and_reshape(model, mid):
  if mid == "mobilenet_v3_small":
    return [model.backbone.features[-1]], None
  if mid == "convnext_tiny":
    return [model.backbone.stages[-1].blocks[-1]], None
  if mid == "vit_base_patch16_224":

    def reshape_transform(tensor):
      b, n, c = tensor.shape
      side = int(np.sqrt(n - 1))
      return (
          tensor[:, 1:, :]
          .reshape(b, side, side, c)
          .permute(0, 3, 1, 2)
          .contiguous()
      )

    return [model.backbone.blocks[-1].norm1], reshape_transform
  raise KeyError(mid)


loaded_models = {mid: load_xai_model(mid) for mid in XAI_MODELS}
positive_target = [ClassifierOutputTarget(0)]

fig, axes = plt.subplots(4, 4, figsize=(8.8, 7.8), dpi=300)

for r_idx, (case_title, t_idx, img_path, true_lbl) in enumerate(selected_cases):
  bgr = cv2.imread(img_path)
  rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
  rgb_resized = cv2.resize(
      rgb, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA
  )

  true_text = "Conjunctivitis" if true_lbl == 1 else "Non-Conjunctivitis"
  axes[r_idx, 0].imshow(rgb_resized)
  axes[r_idx, 0].set_title(
      f"{case_title}\n[Ground Truth: {true_text}]",
      fontsize=7.2,
      fontweight="bold",
  )
  axes[r_idx, 0].axis("off")

  x = eval_transform(image=rgb_resized)["image"].unsqueeze(0).to(DEVICE)

  for c_idx, mid in enumerate(XAI_MODELS, start=1):
    m = loaded_models[mid]
    target_layers, reshape = get_target_layer_and_reshape(m, mid)
    m.zero_grad(set_to_none=True)

    with GradCAM(
        model=m, target_layers=target_layers, reshape_transform=reshape
    ) as cam:
      cam_map = cam(input_tensor=x, targets=positive_target)[0]

    with torch.no_grad():
      prob = torch.sigmoid(m(x)).item()
    tau = benchmark_results[mid]["threshold"]
    pred_text = "Conj." if prob >= tau else "Non-Conj."

    axes[r_idx, c_idx].imshow(rgb_resized)
    axes[r_idx, c_idx].imshow(cam_map, cmap="jet", alpha=0.45)
    axes[r_idx, c_idx].axis("off")
    title_str = (
        f"{XAI_NAMES[mid]}\nPred: {pred_text} ($p$={prob:.2f})"
        if r_idx == 0
        else f"Pred: {pred_text} ($p$={prob:.2f})"
    )
    axes[r_idx, c_idx].set_title(
        title_str, fontsize=7.2, fontweight="bold" if r_idx == 0 else "normal"
    )

plt.tight_layout()
xai_out_path = PLOT_DIR / "figure6_xai_gradcam.png"
plt.savefig(xai_out_path, dpi=300, bbox_inches="tight")
plt.close()

del loaded_models
torch.cuda.empty_cache()
print("Saved balanced binary Grad-CAM figure to:", xai_out_path)
print("\n>>> EXECUTION COMPLETE: Both figures successfully generated. <<<")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 1.7 MB/s eta 0:00:00
>>> Re-generating Figure 4 (Predictive vs. Latency Trade-off)...
Saved non-overlapping Figure 4 to: /kaggle/working/plots/figure4_compute_tradeoff.png

>>> Generating Balanced Binary Explainability (Grad-CAM) across both classes...
Saved balanced binary Grad-CAM figure to: /kaggle/working/plots/figure6_xai_gradcam.png

>>> EXECUTION COMPLETE: Both figures successfully generated. <<<


In [15]:
import shutil
from IPython.display import FileLink

# Package everything in /kaggle/working into a single zip file
shutil.make_archive('/kaggle/working/iccit2026_complete_outputs', 'zip', '/kaggle/working')
print("Outputs zipped successfully! Click the link below to download:")
FileLink(r'iccit2026_complete_outputs.zip')

Outputs zipped successfully! Click the link below to download:


/kaggle/working/iccit2026_complete_outputs.zip